In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from natsort import natsorted
from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODE, NeuralEulerODEPendulum, NeuralEulerODECartpole
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law
from plot_helpers import plot_jsd_model_prediction_relation, plot_model_rollouts

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

## Overview

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "fluid_tank" / "2step",
    model_class=NeuralEulerODE,
    verbose=False,
    expecting_sub_folders=False,
    recompute_jsd=True,
)

In [ ]:
plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "pendulum" / "2step",
    model_class=NeuralEulerODEPendulum,
    verbose=False,
    expecting_sub_folders=False,
    recompute_jsd=True,
)

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

plot_jsd_model_prediction_relation(
    data_path=DataPaths().model_learning_cs_out / "cart_pole" / "2step",
    model_class=NeuralEulerODECartpole,
    verbose=False,
    expecting_sub_folders=False,
    penalty_function=None,
    recompute_jsd=True,
)

In [ ]:
# what is missing?
data_path = DataPaths().model_learning_cs_out / "cart_pole" / "2step"
result_paths = glob.glob(str(data_path) + "/*.eqx")
result_paths = natsorted(result_paths)

n_results = len(result_paths)
print("# or results:", n_results)
print(80 * "-")

# for result_path in tqdm(result_paths, total=n_results):
#     result = ModelExpDataResult.from_file(
#         filename=result_path,
#         model_class=NeuralEulerODECartpole,
#     )

In [ ]:
corresponding_paths = []
exp_results = []
ids = []

for result_path in result_paths:
    result = ModelExpDataResult.from_file(
        filename=result_path,
        model_class=NeuralEulerODECartpole,
    )

    ids.append(result.exp_id)
    exp_results.append(result)
    corresponding_paths.append(result_path)

# print(len(exp_results))

# exp_results_dict = {}

# for result in exp_results:
#     if result.exp_id in exp_results_dict.keys():
#         exp_results_dict[result.exp_id].append(result)
#     else:
#         exp_results_dict[result.exp_id] = [result]

# for exp_id in exp_results_dict.keys():
#     if len(exp_results_dict[exp_id]) > 1:
#         print(exp_id)
            

In [ ]:
ids

In [ ]:
corresponding_paths

In [ ]:
from collections import Counter

In [ ]:
Counter(ids)

In [ ]:
fluid_tank_ids = dict(
    dmpe=[
        '2025-08-27_14-55-49',
        '2025-08-27_14-57-26',
        '2025-08-27_14-59-04',
        '2025-08-27_15-00-41',
        '2025-08-27_15-02-19',
    ],
    pm_dmpe=[
        '2025-08-28_10-31-04',
        '2025-08-28_10-36-03',
        '2025-08-28_10-41-00',
        '2025-08-28_10-45-58',
        '2025-08-28_10-50-54',
    ],
    sgoats=[
        '2025-07-23_17-05-04',
        '2025-07-23_17-25-52',
        '2025-07-23_17-46-10',
        '2025-07-23_18-04-37',
        '2025-07-23_18-25-10',
    ],
    igoats=[
        '2025-07-23_16-44-48',
        '2025-07-23_17-03-43',
        '2025-07-23_17-22-58',
        '2025-07-23_17-40-58',
        '2025-07-23_17-59-50',
    ],
    random_walk=[
        '2025-07-24_11-19-31',
        '2025-07-24_11-19-43',
        '2025-07-24_11-19-54',
        '2025-07-24_11-20-06',
        '2025-07-24_11-20-18',
    ],
)

pendulum_ids = dict(
    dmpe=[
        '2025-08-27_14-57-18',
        '2025-08-27_15-00-42',
        '2025-08-27_15-04-07',
        '2025-08-27_15-07-34',
        '2025-08-27_15-11-00',
    ],
    pm_dmpe=[
        '2025-08-28_10-35-03',
        '2025-08-28_10-44-58',
        '2025-08-28_10-54-58',
        '2025-08-28_11-04-31',
        '2025-08-28_11-14-47',
    ],
    sgoats=[
        '2025-07-23_17-04-21',
        '2025-07-23_17-22-38',
        '2025-07-23_17-40-52',
        '2025-07-23_18-00-08',
        '2025-07-23_18-17-27',
    ],
    igoats=[
        '2025-07-23_16-44-37',
        '2025-07-23_17-02-47',
        '2025-07-23_17-21-38',
        '2025-07-23_17-40-06',
        '2025-07-23_17-59-04',
    ],
    random_walk=[
        '2025-07-24_11-15-20',
        '2025-07-24_11-15-32',
        '2025-07-24_11-15-44',
        '2025-07-24_11-15-55',
        '2025-07-24_11-16-06',
    ],
)

cart_pole_ids = dict(
    dmpe=[
        '2025-08-27_15-13-55',
        '2025-08-27_15-34-19',
        '2025-08-27_15-54-39',
        '2025-08-27_16-15-02',
        '2025-08-27_16-35-27',
    ],
    pm_dmpe=[
        '2025-08-28_10-44-48',
        '2025-08-28_11-04-06',
        '2025-08-28_11-23-21',
        '2025-08-28_11-42-41',
        '2025-08-28_12-02-02',
    ],
    sgoats=[
        '2025-07-24_11-52-44',
        '2025-07-24_12-31-52',
        '2025-07-24_13-10-21',
        '2025-07-24_13-50-30'
        '2025-07-24_14-28-45',
    ],
    igoats=[
        '2025-07-23_17-34-12',
        '2025-07-23_18-35-44',
        '2025-07-23_19-39-24',
        '2025-07-23_20-44-08',
        '2025-07-23_21-51-53',
    ],
    random_walk=[
        '2025-07-24_11-19-00',
        '2025-07-24_11-19-13',
        '2025-07-24_11-19-26',
        '2025-07-24_11-19-39',
        '2025-07-24_11-19-52',
    ],
)

def get_algo(exp_id, ids):
    for algo_name, exp_ids_in_algo in ids.items():
        if exp_id in exp_ids_in_algo:
            return algo_name

get_fluid_tank_algo = partial(get_algo, ids=fluid_tank_ids)
get_pendulum_algo = partial(get_algo, ids=pendulum_ids)
get_cart_pole_algo = partial(get_algo, ids=cart_pole_ids)

In [ ]:
get_fluid_tank_algo(

In [ ]:
cart_pole_ids

- sort each of the unique ids to an algorithm (it is always the first 5 entries for each system excitation experiment):

In [ ]:
for result in natsorted(load_all_experiment_results(DataPaths().model_learning_experiments / "various_exp_together_in" / "fluid_tank", None)):
    print(result["exp_id"])
    plot_sequence(result["observations"], result["actions"], env.tau, env.obs_description, env.action_description)
    plt.show()

In [ ]:
ids

In [ ]:
exp_results

In [ ]:
for exp_id in exp_results_dict.keys():
    if len(exp_results_dict[exp_id]) > 1:
        print(exp_id, ":", len(exp_results_dict[exp_id]))
            

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[-1],
    model_class=NeuralEulerODECartpole,
)

In [ ]:
jnp.arange(1000, 15001, 1000).shape

In [ ]:
env, _, _, _ = setup_cart_pole_env()
for result in load_all_experiment_results(DataPaths().model_learning_experiments / "various_exp_together_in" / "pendulum", None):
    print(len(result["observations"]))
    print(len(result["actions"]))

## Rollout plots:

In [ ]:
from dmpe.related_work.random_walk import random_walk_control_law
from dmpe.evaluation.model_evaluation import RolloutComparison
from plot_helpers import plot_model_rollouts

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "fluid_tank" / "10step_large") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[0],
    model_class=NeuralEulerODE,
)

env, penalty_function, featurize, _ = setup_fluid_tank_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    model=result.median_model,
    batch_size=10,
    sequence_length=100,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "pendulum" / "10step_large") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[0],
    model_class=NeuralEulerODEPendulum,
)

env, penalty_function, featurize, _ = setup_pendulum_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=lambda x: x,
    model=result.median_model,
    batch_size=10,
    sequence_length=400,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "cart_pole" / "10step_large") + "/*.eqx")

result = ModelExpDataResult.from_file(
    filename=result_paths[1],
    model_class=NeuralEulerODECartpole,
)

env, penalty_function, featurize, _ = setup_cart_pole_env()
plot_model_rollouts(
    env=env,
    penalty_function=penalty_function,
    featurize=lambda x: x, #featurize,
    model=result.median_model,
    batch_size=5,
    sequence_length=100,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
result.visualize_training()

## Rollout comparison:

In [ ]:
from plot_helpers import plot_jsd_model_rollout_relation
from dmpe.related_work.random_walk import random_walk_control_law

In [ ]:
env, penalty_function, featurize, _ = setup_fluid_tank_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "fluid_tank"  / "10step_large",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODE,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
env, penalty_function, featurize, _ = setup_pendulum_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "pendulum" / "10step_large",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODEPendulum,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

plot_jsd_model_rollout_relation(
    data_path=DataPaths().model_learning_cs_out / "cart_pole" / "10step_large",
    env=env,
    penalty_function=penalty_function,
    featurize=featurize,
    batch_size=400,
    sequence_length=10,
    model_class=NeuralEulerODECartpole,
    key=jax.random.PRNGKey(0),
    control_law=partial(random_walk_control_law, n_tries=4000),
)

## Indepth:

### Fluid tank:

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "fluid_tank" / "2step" ) + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

In [ ]:
env, _, featurize, _ = setup_fluid_tank_env()
wrapped_env = EnvWrapper(env, featurize=lambda x: x)

points_per_dim = 100

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=1,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[-10],
    model_class=NeuralEulerODE,
)
print(result.exp_id)

result.visualize_training()

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[55], # 2
    model_class=NeuralEulerODE,
)
print("Number of datapoints in underlying dataset:", result.n_datapoints)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=wrapped_env.featurize),
    model_evaluator,
    labels=["h", "q_in"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=1, c="crimson")

plt.show()


# for model in result.models:

#     fig, axs = result.visualize_model_prediction_performance(
#         NodeModelWrapper(model, featurize=wrapped_env.featurize),
#         model_evaluator,
#         labels=["h", "q_in"],
#     )
    
#     data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
#     n_features = data_points.shape[-1]
    
#     for i in range(n_features):
#         for j in range(n_features):
#             axs[j, i].scatter(data_points[..., i], data_points[..., j], s=1, c="crimson")
    
#     plt.show()

In [ ]:
from dmpe.utils.density_estimation import build_grid_2d

def test(wrapped_model, wrapped_env, grid, env):

    obs = grid[:, :1]
    act = grid[:, 1:]
    
    model_pred = eqx.filter_vmap(wrapped_model.step, in_axes=(0, 0, None))(obs, act, env.tau)
    # model_pred = eqx.filter_vmap(result.median_model.func, in_axes=(0, 0))(obs, act)

    #model_pred = eqx.filter_vmap(mlp, in_axes=(0))(grid)#jnp.concatenate([obs, act], axis=-1))
    
    env_pred = eqx.filter_vmap(wrapped_env.step, in_axes=(0, 0, None))(obs, act, env.tau)
    return model_pred - env_pred

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[-1], # 2
    model_class=NeuralEulerODE,
)
print("Number of datapoints in underlying dataset:", result.n_datapoints)
wrapped_model = NodeModelWrapper(result.median_model, featurize=featurize)

difference_map, _ = model_evaluator.default_metrics["pred_comp"](
    wrapped_model,
    model_evaluator.gt_model,
)
n_features = model_evaluator.obs_dim + model_evaluator.act_dim
reshaped_difference_map = difference_map.reshape(
    [model_evaluator.validation_points_per_dim] * n_features
)
# abs_map = jnp.mean(jnp.abs(reshaped_difference_map) ** 2, axis=-1)
print(jnp.abs(reshaped_difference_map).shape)
abs_map = jnp.abs(reshaped_difference_map[:, :]) # jnp.linalg.norm(reshaped_difference_map, axis=-1)
z = model_evaluator.constraint_data_space_grid

grid_len_per_dim = int(np.sqrt(z.shape[0]))
z_plot = z.reshape((grid_len_per_dim, grid_len_per_dim, 2))

fig, axs = plt.subplots(1, 1, figsize=(9, 9))
 
cax = axs.contourf(
    z_plot[..., 0],
    z_plot[..., 1],
    abs_map,
    antialiased=False,
    levels=50,
    alpha=0.9,
    cmap=plt.cm.coolwarm,
)
data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
axs.scatter(data_points[..., 0], data_points[..., 1], s=1, c="crimson")
axs.set_xlabel(r"$h$")
axs.set_ylabel(r"$q$")
plt.show()

#################################################################################################################

grid = build_grid_2d(-1, 1, 10)
out = test(wrapped_model, wrapped_env, grid, env)  # 0.006

fig = plt.figure(figsize=(6, 6))
axs = fig.add_subplot(111, projection="3d")

_ = axs.plot_surface(
    z_plot[..., 0],
    z_plot[..., 1],
    abs_map,
    antialiased=False,
    alpha=0.5,
    cmap=plt.cm.coolwarm,
)

axs.scatter(
    grid[:, 0],
    grid[:, 1],
    jnp.squeeze(jnp.abs(out)),
    c="r"
)

axs.set_xlabel(r"$h$")
axs.set_ylabel(r"$q$")
plt.show()

### Pendulum

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "pendulum" / "2step") + "/*.eqx")

In [ ]:
env, _, featurize, _ = setup_pendulum_env()
wrapped_env = EnvWrapper(env, featurize=featurize)

points_per_dim = 100

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=2,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

In [ ]:
result = ModelExpDataResult.from_file(
    filename=result_paths[250], # 2
    model_class=NeuralEulerODEPendulum,
)

print(result.n_datapoints)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize),
    model_evaluator,
    labels=["theta", "omega", "torque"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)

plt.show()

### Cart-pole:

In [ ]:
from dmpe.utils.sets.shared import load_discretized_set

In [ ]:
# S_xu = load_discretized_set(
#     DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json",
# )

In [ ]:
result_paths = glob.glob(str(DataPaths().model_learning_cs_out / "cart_pole" / "2step") + "/*.eqx")
print("# of results: ", len(result_paths))
result_paths = natsorted(result_paths)

In [ ]:
env, penalty_function, featurize, _ = setup_cart_pole_env()

# featurize = lambda x: x
wrapped_env = EnvWrapper(env, featurize=featurize)

points_per_dim = 20

model_evaluator = ModelEvaluator(
    constraint_function=default_constraint_function,
    gt_model=wrapped_env,
    obs_dim=4,
    act_dim=1,
    validation_points_per_dim=points_per_dim,
    tau=env.tau,
)

result = ModelExpDataResult.from_file(
    filename=result_paths[41],
    model_class=NeuralEulerODECartpole,
)

fig, axs = result.visualize_model_prediction_performance(
    NodeModelWrapper(result.median_model, featurize=featurize),
    model_evaluator,
    labels=["d", "v", "theta", "omega", "u"],
)

data_points = jnp.concatenate([result.observations, result.actions], axis=-1)
n_features = data_points.shape[-1]

for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
axs[j, i].grid(True)
axs[j, i].set_xlim(-1.1, 1.1)
axs[j, i].set_ylim(-1.1, 1.1)


plt.savefig(f"qual_model_perf_data_relationship.png", dpi=400, bbox_inches='tight');
plt.show()

# featurize = lambda x :x
#
# plot_model_rollouts(
#     env=env,
#     penalty_function=penalty_function,
#     featurize=featurize,
#     model=result.median_model,
#     batch_size=10,
#     sequence_length=100,
#     key=jax.random.PRNGKey(0),
#     control_law=partial(random_walk_control_law, n_tries=4000),
# )
# plt.show()

In [ ]:
fig, axs = S_xu.visualize()
for i in range(n_features):
    for j in range(n_features):
        axs[j, i].scatter(data_points[..., i], data_points[..., j], s=0.25, c="r")
        axs[j, i].set_xlim(-1.1, 1.1)
        axs[j, i].set_ylim(-1.1, 1.1)

In [ ]:
data_points.shape